# Agents, Session 1: Foundations of Agents

## The concierge that could talk but not act

Prasad's week starts well. DilSeVector's concierge, the one you built over the RAG sessions, answers every coffee question his users throw at it. Then, on Monday morning, he types a message that is not a question at all:

> "Book a table for two at the pink café in Dadar this Saturday at 7 pm and add it to my calendar."

The concierge replies politely. Nothing is booked. Nothing is in his calendar.

This session is about why that happens and what it takes to fix it. By the end you will have written, in Python with Gemini:

- an LLM that understands intent and returns structured data,
- an LLM that remembers, one that plans and replans, and one that improves from feedback,
- a real agent loop, first by hand and then with LangChain's `create_agent`,
- the four workflow patterns: evaluator-optimizer, parallelization, chaining and routing,
- a travel planner for Prasad's first trip, exported to his calendar.

![The story so far](attachment:story.png)

*The story so far, and where today begins.*

### Before you start

Run the cells in order. The whole notebook makes about 55 Gemini calls. The free tier allows 5 calls a minute, so the setup cell adds a rate limiter that makes calls wait their turn instead of failing. Some cells will pause for a few seconds. That is expected.

In [ ]:
%pip install -q "langchain>=1.0" "langchain-google-genai>=4.0" icalendar requests

In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

# At most 4 calls a minute (the free tier allows 5), shared by everything in this notebook
rate_limiter = InMemoryRateLimiter(requests_per_second=4 / 60, check_every_n_seconds=0.5, max_bucket_size=1)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    rate_limiter=rate_limiter,
)

reply = llm.invoke("In one sentence, greet a class that is about to learn about AI agents.")
print(reply.text)

## 1. Story Till now: the concierge that could talk but not act

### Where we have come from

In the prompting module you treated the LLM as a very capable brain that answers well only if you ask the right way. Zero-shot, few-shot, chain of thought and ReAct were all ways of asking. Whatever the technique, the shape was the same: text goes in, text comes out.

![Prompt in, text out](attachment:llm_basic.png)

*An LLM on its own: a prompt goes in, generated text comes out.*

That brain had a frozen memory. Its knowledge stops at its training date, and it has never seen your private data. So in the RAG module you let it look things up: your code retrieves the relevant pages and places them in the prompt, and the model answers from them. It was like giving an intern access to the internet and your filing cabinet. DilSeVector's concierge is exactly this.

![Retrieval-augmented generation](attachment:rag.png)

*RAG: your data is indexed; the user's query pulls relevant pieces into the prompt.*

### The limit neither module removed

Both pictures end in the same place: text. Let us send Prasad's two requests to Gemini exactly as the concierge would. His calendar is an ordinary Python list, and it starts empty.

**Before you run it:** after both replies, how many events will be in `prasads_calendar`?

In [ ]:
prasads_calendar = []

prasads_requests = [
    "Book a table for two at the pink café in Dadar this Saturday at 7 pm and add it to my calendar.",
    "Find the cheapest IndiGo flight from Mumbai to Kullu next Friday and book it for me.",
]

for request in prasads_requests:
    reply = llm.invoke(request)
    print("Prasad:", request)
    print("Gemini:", reply.text)
    print("-" * 60)

print("Events in Prasad's calendar:", len(prasads_calendar))

The calendar is empty. Sometimes the model says honestly that it cannot book anything; sometimes it writes something that sounds like a confirmation. The second case is worse: Prasad would turn up at the café expecting a table.

LLMs are **reactive**. They do not start anything on their own, they have no goals of their own, and they act only when a prompt arrives, and even then the only thing they produce is text. A calculator is reactive in the same way: it never wakes up and decides to add 2 and 2; it reacts when you press the keys. An LLM will never decide to book your flight. It predicts the next words of a reply.

So what exactly is missing? Three things, and the rest of this session builds each one.

1. **Hands.** Changing a calendar needs code to run. We will give the model *tools*, Python functions it can ask us to run.
2. **Eyes on the present.** The model does not know today's date, so "this Saturday" means nothing to it. A tool can tell it.
3. **A second look.** Prasad's request has a hidden condition: book only if he is free. That needs one step (check the calendar), a decision based on the result, then another step. A single call cannot react to a result it has not seen. We need a *loop*.

Think of how you use Siri or Google Assistant. You say "set an alarm for 7 am" and it is set. You did not open the clock app and tap through menus; the assistant understood the request, chose the right action and carried it out. That is the idea behind an AI agent. Let us define it properly.

## 2. What an agent is

An **agent** is anything that can observe what is happening around it, make decisions, and take actions to achieve a goal. A robot vacuum senses dirt and walls, works out where to go and cleans. A smart thermostat reads the temperature, decides whether to heat or cool, and switches the AC on. A video game character watches what you do and reacts, plans and moves.

What makes these agents special is the combination of abilities in the picture below, arranged as a loop.

![The agent loop](attachment:agent_loop.jpg)

*The agent loop, from the session document.*

**Perceive.** The agent observes its environment: a sensor value, a user's message, the result of a tool. Without perception an agent is driving blindfolded.

**Reason and plan.** It interprets what it perceived and chooses the best next step toward its goal. Sensing alone is not enough; the vacuum must decide to go left or right, the game character must decide to attack or run.

**Act.** It carries out the decision, and the world changes. Deciding is useless without doing.

**Feedback.** It checks the outcome. Did the vacuum actually clean the spot, or did it miss half the room? Did the AC overcool? Without checking, an agent repeats its mistakes. The vacuum bumps a wall and changes direction; the thermostat reads the new temperature and adjusts; a self-driving car changes lanes, and its sensors confirm whether the gap is safe.

Then it goes round again. An agent is not a one-and-done system that perceives once, acts once and stops. It keeps going until the goal is met.

### Goal-driven and proactive

Two properties come up whenever agents are discussed.

**Goal-driven** means you state the end result, not the steps. Tell a smart travel assistant "I want to plan a trip to Japan next month" and every choice it makes, checking flights, comparing hotels, suggesting activities, points at that goal. It behaves like a car's GPS: once you type the destination, it keeps rerouting toward it.

**Proactive** means it takes initiative. A passive app waits for "search flights", then "search hotels", then "check weather". A proactive one says: "Flights are cheaper if you leave two days earlier; should I change the dates?", "Rain is expected in Kyoto; shall I suggest indoor activities?", "Your passport expires soon; renew it before booking."

With an LLM, much of this behaviour starts in the instructions you give it. Prasad is planning a December trip with his match. Here is the same request to two assistants whose only difference is the system prompt.

**Before you run it:** what do you expect the proactive assistant to raise that the passive one will not?

In [ ]:
trip_request = "I want to plan a trip to Manali with my partner in the last week of December."

assistant_styles = {
    "Passive": "You are a travel assistant. Answer exactly what the user asks and nothing more. Keep it under 80 words.",
    "Proactive": (
        "You are a proactive travel assistant. Keep the user's end goal in mind. Before answering, think about "
        "what could go wrong or what they have not asked about yet (season, weather, road conditions, bookings "
        "that sell out, documents, budget). Raise the two most important points as questions. Keep it under 80 words."
    ),
}

for style in assistant_styles:
    instructions = assistant_styles[style]
    reply = llm.invoke([SystemMessage(content=instructions), HumanMessage(content=trip_request)])
    print(f"=== {style} assistant ===")
    print(reply.text)
    print()

The proactive assistant usually raises snow, road closures on the way to Manali, or holiday-season prices, without being asked. Notice that it still did not *do* anything: it cannot check the weather or book a room. Proactive instructions shape what the model says. Tools and a loop, which come in Chapter 4, let it act.

### From agents to AI agents

An **AI agent** is an agent whose perceive and decide steps are powered by AI: it understands natural language, plans several steps, and uses digital tools and apps on its own. Picture a student who says "I need to finish all my assignments this week." A simple agent reminds them of deadlines. An AI agent reads the chapters, summarises the key points, builds a study schedule and sets a reminder for each task. The fictional ideal is J.A.R.V.I.S. from Iron Man: told "get my coffee ready", it plans and delivers without step-by-step instructions.

So AI agents are goal-oriented and proactive, understand natural language, plan multi-step actions, and interact with tools, apps and external systems. They are agents on steroids. But they did not appear overnight.

## 3. How agents evolved, rebuilt with an LLM

Agents grew smarter in stages, and each stage fixed a limit of the one before. The loop stayed the same throughout. What changed was **who decides the next step, and how**. That question is the thread of this whole session: keep asking it and every new idea will slot into place.

![Evolution of agents](attachment:evolution.png)

*Six stages of agents, each answering "who decides?" differently.*

For each stage we tell its story, and then write the same idea as GenAI code. You will see that every old idea has a modern LLM form you will use constantly.

### 3.1 Rule-based agents: the obedient robot (1950s to 1970s)

The earliest agents did exactly what they were told and nothing more. Imagine a robot that washes dishes only when you put a cup on one particular spot. It never improvises.

They worked on if-then rules: if this happens, do that. **ELIZA** (1966) was one of the first chatbots. It imitated a therapist by pattern matching ("I feel sad" became "Why do you feel sad?"). People were amazed, but ELIZA understood nothing. A **thermostat** that turns the heater on below 20°C and off above 25°C is the same idea in hardware.

From a developer's point of view, a rule-based agent is simply a script of hardcoded rules:

```python
if temperature < 20:
    turn_heater_on()
elif temperature > 25:
    turn_heater_off()
```

It works perfectly for the cases you coded and does nothing sensible for anything else. These agents mattered because they showed that rule-following programs could automate real tasks and introduced the idea of a program with a goal. Their limits were just as clear: they could not handle surprises, could not learn, and needed one rule for every condition.

Before DilSeVector, the pink café ran a WhatsApp booking bot built exactly this way. Here it is, with three real messages from customers.

In [ ]:
cafe_bot_rules = {
    "book": "Please send the date, time and number of guests.",
    "menu": "Our menu is at pinkcafe.example/menu",
    "timing": "We are open from 8 am to 11 pm.",
}

def cafe_bot_reply(message):
    lowered = message.lower()
    for keyword in cafe_bot_rules:
        if keyword in lowered:
            return cafe_bot_rules[keyword]
    return "Sorry, I did not understand. Please call us."

customer_messages = [
    "Can I book a table for 2 this Saturday at 7?",
    "My date doesn't drink coffee after 5, do you have anything without caffeine?",
    "Is it quiet enough on a Saturday evening to actually talk?",
]

for message in customer_messages:
    print("Customer:", message)
    print("Rule bot:", cafe_bot_reply(message))
    print()

Only the message containing the keyword "book" worked. The other two are perfectly reasonable questions, and the bot has no rule for either.

**The same idea with an LLM.** Instead of matching keywords, we ask Gemini to *understand* each message and return the answer as structured data our code can act on. `with_structured_output` takes a JSON schema describing the shape we want back, and Gemini's reply arrives as a Python dictionary. This is the modern "perceive" step: turning messy human language into something a program can use.

**Before you run it:** what intent will Gemini give the caffeine question?

In [ ]:
understanding_schema = {
    "title": "message_understanding",
    "description": "What each café customer wants",
    "type": "object",
    "properties": {
        "messages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "message": {"type": "string"},
                    "intent": {"type": "string", "enum": ["booking", "menu_question", "ambience_question", "other"]},
                    "details": {"type": "string", "description": "The specific need, in a few words"},
                },
                "required": ["message", "intent", "details"],
            },
        }
    },
    "required": ["messages"],
}

understanding_llm = llm.with_structured_output(understanding_schema)

all_messages_text = ""
for message in customer_messages:
    all_messages_text = all_messages_text + "- " + message + "\n"

result = understanding_llm.invoke("Understand each of these café customer messages:\n" + all_messages_text)

for item in result["messages"]:
    print(f"{item['intent']:18s} | {item['details']}")
    print(f"{'':18s} | from: {item['message']}")

One call understood all three, including the two the rule bot could not handle, and the result is a dictionary your code can route on: `booking` goes to the booking flow, `menu_question` to the menu, and so on. You did not write a rule for "caffeine" or "quiet". That is the leap from Stage 1 to today in a single cell.

### 3.2 Reactive agents: the alert machine (1980s to 1990s)

Life is not predictable, and rule-based agents froze when something happened that had no rule. Reactive agents shifted from "do this if that happens" to "observe what is happening and act accordingly". They sense their environment continuously and respond at once, but they do not plan ahead and do not learn.

![A reactive agent](attachment:reactive.png)

*A reactive agent: sensors feed an observation, rules select an action, and the action changes the environment.*

Robot vacuums detect walls and furniture and change direction. Lab robots follow light or avoid obstacles. Simple game characters attack or move when you do. Suddenly machines felt alive, because they coped with a changing world instead of waiting for exact instructions.

For a developer, a reactive agent is an event-driven program: it listens for events and responds immediately.

```python
def on_obstacle_detected():
    change_direction()

def on_light_detected():
    follow_light()
```

The limitation: reactive agents cannot plan multi-step actions and cannot learn. They are reactive but not strategic, and they have no memory of what happened a moment ago.

**The same idea with an LLM.** A bare LLM call is a reactive agent. It responds to the message in front of it and remembers nothing. Prasad tells the model something about himself, then asks about it in the next call.

**Before you run it:** will the second call know his favourite drink?

In [ ]:
first_message = "Hi, I'm Prasad. I run a dating app called DilSeVector and my favourite drink is kesar cold coffee."
second_message = "What's my name, and what's my favourite drink?"

first_reply = llm.invoke(first_message)
print("Call 1 reply:", first_reply.text)

second_reply = llm.invoke(second_message)
print("\nCall 2 reply:", second_reply.text)

The second call has no idea. Each call is independent: the model reacts to its input and forgets everything afterwards.

The fix is not a smarter model. It is **memory you manage yourself**: keep the conversation in a list and send the whole list every time. This is exactly how every chat app works behind the scenes.

In [ ]:
conversation = [
    HumanMessage(content=first_message),
    first_reply,                             # the model's own earlier answer
    HumanMessage(content=second_message),
]

reply_with_memory = llm.invoke(conversation)
print("With the conversation sent along:", reply_with_memory.text)
print(f"\nMessages sent in this call: {len(conversation)}")

Now it knows. The model did not change; the prompt did. Keep this in mind: in every agent you build, "memory" means *what you choose to send back to the model*. Session 5's memory RAG used the same trick.

### 3.3 Deliberative agents: the strategic player (1990s to 2000s)

Some tasks need more than reflexes. A robot that must deliver a package across a cluttered office cannot just react to each obstacle; it needs a strategy. Deliberative agents think ahead: they plan a sequence of actions and choose the best path to the goal. Chess engines look several moves ahead. Navigation systems plan routes around traffic. Warehouse robots plan paths to pick many items without collisions.

![A deliberative agent](attachment:deliberative.png)

*A deliberative agent adds an internal state, its model of the world, between observing and acting.*

Take a delivery robot that must carry a package from Storage Room A to Office 210. It works through five parts.

**1. Internal model of the world.** It holds a map of the office: rooms, walls, lifts, its own position. It can simulate a route without moving, so it thinks about the future without touching the real world.

**2. Goal.** An explicit objective guides every decision: deliver the package to Office 210 in the shortest time, with sub-goals such as avoiding obstacles and taking the lift if needed.

**3. Planning.** It enumerates possible actions, predicts their outcomes using the map, evaluates sequences against the goal, and selects the best plan. Common methods are *search-based planning* (breadth-first search explores every route of length 1, then 2, and so on; depth-first search follows one route as far as possible before backing up; A* explores the most promising routes first using an estimate of the distance left), *rule-based planning* (if certain conditions hold, choose a set action), and *hierarchical planning* (split the goal into sub-goals such as "move to the lift → ride to floor 2 → walk the corridor → deliver", and plan each one).

**4. Reasoning.** With several possible plans, it reasons about which is feasible and which is best. *Logic-based reasoning* derives conclusions from facts and rules: "the lift is out of service; any plan using the lift is infeasible; so take the stairs." *Utility-based reasoning* scores options by expected benefit: path 1 is shorter but crowded, path 2 is slightly longer but clear, and the robot picks path 2 because the expected total time is lower.

**5. Execution with monitoring.** It carries out the plan step by step and keeps watching. If a person steps into the corridor, it replans on the fly.

In code you might picture it as a set of helper functions: `map_environment()` builds the model, `define_goal()` sets the objective, `plan_path()` chooses the route, `evaluate_options()` weighs trade-offs, `execute_plan()` moves step by step, and `monitor_and_replan()` handles surprises.

> **AlphaGo and AlphaZero.** DeepMind's AlphaGo shocked the world by beating the best human Go players, but it was built for Go alone. AlphaZero (2017) asked a bigger question: could one algorithm learn *any* such game with no human examples? Given only the rules, it played millions of games against itself, using the win or loss of each as feedback. A neural network supplied its intuition about which moves looked promising, and Monte Carlo Tree Search supplied calculation by exploring future positions. It mastered Go, chess and shogi. In chess it overtook Stockfish, the strongest engine of the time, after about four hours of training; in shogi it overtook Elmo, Japan's strongest program, in about two. Grandmasters described its play as creative and bold. The same ideas went on to science: DeepMind's AlphaFold predicts protein structures. AlphaZero combines deliberation (tree search) with learning (self-play), which leads straight to the next stage.

Deliberative agents were a big leap, but they had two problems. They thought slowly: by the time a plan was ready, the world might have changed, and replanning everything repeatedly is expensive. And they knew only what their map and rules told them: drop the robot into a new building and it is clueless, and every delivery is like its first.

**The same idea with an LLM.** An LLM can be asked to *plan before acting*: produce an ordered list of steps from a description of the world and the tools available, without executing anything. We ask for the plan as structured data so our code could later carry out each step. Prasad's goal is a dinner date at the pink café this weekend.

In [ ]:
plan_schema = {
    "title": "plan",
    "description": "An ordered plan to reach a goal",
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "action": {"type": "string", "description": "What to do in this step"},
                    "tool": {"type": "string", "description": "Which tool the step uses, or 'none'"},
                },
                "required": ["action", "tool"],
            },
        }
    },
    "required": ["steps"],
}
planner_llm = llm.with_structured_output(plan_schema)

world_model = """What you know about the world:
- Tools available: get_today, check_calendar, call_cafe (books a table by phone), add_calendar_event, send_whatsapp.
- Prasad is busy this Saturday from 18:00 to 21:00 (office cricket match).
- The pink café in Dadar is open 8:00 to 23:00 and takes bookings by phone only."""

date_goal = "Arrange a dinner date for Prasad and his match at the pink café this weekend."

plan = planner_llm.invoke(world_model + "\n\nGoal: " + date_goal + "\nMake a step-by-step plan. Do not carry anything out.")

print("Plan:")
step_number = 1
for step in plan["steps"]:
    print(f"  {step_number}. {step['action']}   [tool: {step['tool']}]")
    step_number = step_number + 1

The model planned in its head, the way the delivery robot simulates a route on its map. Now the world changes. Suppose the café is called and says it is closed on Sunday for a private event. A deliberative agent must notice and **replan**.

**Before you run it:** what will the new plan do about Saturday's cricket match?

In [ ]:
old_plan_text = ""
for step in plan["steps"]:
    old_plan_text = old_plan_text + "- " + step["action"] + "\n"

surprise = "call_cafe returned: 'Sorry, we are closed all day this Sunday for a private event.'"

new_plan = planner_llm.invoke(
    world_model + "\n\nGoal: " + date_goal
    + "\n\nYour earlier plan was:\n" + old_plan_text
    + "\nWhile executing it, this happened: " + surprise
    + "\nMake a revised step-by-step plan from this point. Do not carry anything out."
)

print("Revised plan:")
step_number = 1
for step in new_plan["steps"]:
    print(f"  {step_number}. {step['action']}   [tool: {step['tool']}]")
    step_number = step_number + 1

The revised plan typically moves the date to Saturday after the match, or to a later day, or asks Prasad to choose. That is monitoring and replanning: the plan met reality, reality pushed back, and the plan changed. This "plan, then execute, then replan" shape is a real pattern for LLM agents. Its weakness is the same as the robot's: the plan is only as good as the model of the world it was given.

### 3.4 Learning agents: the eager student

Where deliberative agents think ahead, **learning agents grow smarter with experience**. Upgrade the delivery robot so it learns from mistakes: it gets stuck in a corridor and avoids it next time, finds shortcuts after several deliveries, and adapts to a new building without anyone rewriting its code.

Every learning agent has four parts.

**Learning element**, the brain's coach. It looks at performance and asks how to do better next time: if the robot keeps bumping into a chair on route A, it updates its knowledge to avoid it.

**Performance element**, the doer. It takes actions using whatever the agent knows right now, even if that knowledge is incomplete.

**Critic**, the feedback system. It compares what happened with what was expected: "the delivery took 15 minutes instead of 5; not good", or "no collisions and fast; great job".

**Problem generator**, the curiosity engine. It pushes the agent to try something new. Route A may be safe but slow; trying route B risks failure but might reveal a much faster path.

Put together: the robot follows a route and is delayed; the critic says it was too slow; the learning element notes that route A has many obstacles; the problem generator suggests route B next time; the performance element takes it and delivers faster. Each delivery makes it smarter.

**Reinforcement learning (RL)** is the modern framework for this idea, and it is how AlphaGo, walking robots and delivery route optimisation at companies like Amazon and Swiggy learn. It is also how you learn a new video game: at first you keep failing (performance element), the game gives points or "Game Over" (critic), you try jumping instead of running (problem generator), and over time you find a winning strategy (learning element).

| Learning-agent part | Reinforcement-learning name | Delivery robot |
|---|---|---|
| Learning element | Policy plus learning algorithm | Updates which route it prefers |
| Performance element | The agent's actions | Moves left, right, forward |
| Critic | Reward signal | +10 for on time, -5 for bumping a chair |
| Problem generator | Exploration versus exploitation | Try risky route B, or stick with safe route A? |

**The same idea with an LLM, and an important honesty point.** Gemini's weights never change while your code runs. An LLM agent "learns" only by *storing lessons and putting them back into its prompt*. That is still genuinely useful. Below, the concierge drafts a WhatsApp message for Prasad. Prasad is the critic. A second LLM call is the learning element: it turns his feedback into general rules. A plain Python list is the agent's memory of lessons. Then we give it a *different* task and see whether the lessons carry over.

In [ ]:
drafting_instructions = "You write short WhatsApp messages on behalf of Prasad, who runs DilSeVector."
lessons = []      # the agent's memory of what it has learned

def instructions_with_lessons():
    text = drafting_instructions
    if len(lessons) > 0:
        text = text + "\nLessons from Prasad's earlier feedback. Always follow them:\n"
        for lesson in lessons:
            text = text + "- " + lesson + "\n"
    return text

# Performance element: first attempt
first_task = "Write a message inviting my match to the pink café in Dadar this Saturday at 9:30 pm."
attempt_1 = llm.invoke([SystemMessage(content=instructions_with_lessons()), HumanMessage(content=first_task)])
print("Attempt 1:\n" + attempt_1.text)

In [ ]:
# Critic: Prasad's feedback
prasads_feedback = ("Too long and too formal, she'll laugh at me. Also she's vegetarian and never drinks "
                    "coffee late at night, so don't suggest the espresso or anything with chicken.")

# Learning element: turn the feedback into general rules for the future
lesson_prompt = ("Turn this feedback into at most three short, general rules for writing future messages. "
                 "Return one rule per line and nothing else.\nFeedback: " + prasads_feedback)
lesson_reply = llm.invoke(lesson_prompt)

for line in lesson_reply.text.split("\n"):
    cleaned = line.strip().lstrip("-*0123456789. ").strip()
    if cleaned != "":
        lessons.append(cleaned)

print("Lessons stored in memory:")
for lesson in lessons:
    print(" -", lesson)

**Before you run it:** the next task is different: a food walk, not a café. Will the lessons still shape the message?

In [ ]:
second_task = "Write a message asking her to join a street-food walk in Girgaon next Sunday evening."
attempt_2 = llm.invoke([SystemMessage(content=instructions_with_lessons()), HumanMessage(content=second_task)])
print("Attempt 2, with lessons:\n" + attempt_2.text)
print(f"\nLessons used: {len(lessons)}. Clear the list and the agent forgets them all.")

The second message is shorter and more casual, and it steers away from non-vegetarian food and late-night coffee, although nobody repeated that feedback. That is learning in the only sense an LLM agent has: feedback turned into memory that shapes the next prompt. Delete `lessons` and it is gone. Keep them in a database and the agent "remembers" across weeks. This is what people usually mean when they say an AI agent adapts to your preferences.

### 3.5 Domain-specific intelligent agents: experts inside a fence

Every stage so far hit the same walls. Classical agents were narrow (good at one job), rule-heavy (choking on complexity), bad at human language (you had to speak their syntax) and rigid when surprised. They were like robots stuck in boxes: each box had its own expertise, but none could step out or talk to the others.

Domain-specific intelligent agents were the bridge to modern AI. They learned from data, handled particular tasks autonomously, and could interact with people in simple ways, but only inside their domain. They had four characteristic capabilities.

**Task-specific expertise.** They excelled at one or a few tasks using pattern recognition and machine-learning models: Netflix and Amazon recommendations, AlphaGo for Go, Siri for alarms and reminders.

**Early learning and adaptation.** They adapted to patterns in data. Netflix refines its suggestions from your viewing; AlphaGo improved over millions of simulated games. But the learning stayed in its domain: a Netflix recommender cannot suddenly play Go.

**Limited natural language.** Some understood basic commands. Siri in 2011 could set an alarm for 7 am, but not "book a hotel and plan flights for next month under $5000".

**Automation within boundaries.** They could run multi-step tasks inside one domain, like warehouse robots navigating shelves or assistants scheduling appointments, but struggled with anything that needed cross-domain reasoning or new tools.

The session document's example is an airline chatbot called **FlyBot**, and Prasad is about to meet it. His December trip starts with a flight from Mumbai to Kullu through Delhi. FlyBot was built the classic way: a developer coded every rule by hand.

In [ ]:
flybot_rules = {
    "book a flight": "Sure. Please tell me your origin, destination and travel date.",
    "change my booking": "Please share your booking reference number.",
    "baggage allowance": "You can find our baggage policy at airline.example/baggage.",
}

def flybot_reply(message):
    lowered = message.lower()
    for keyword in flybot_rules:
        if keyword in lowered:
            return flybot_rules[keyword]
    if "cancel" in lowered:
        return "To rebook a cancelled flight, please visit airline.example/rebook."
    return "I'm sorry, I can only assist with flight-related queries."

cancellation_message = (
    "Hi, my connecting flight from Delhi to Kullu was just canceled due to a snowstorm warning, and I'll miss "
    "the start of my trip. I need to be rebooked on the next available flight, and you need to make sure I have "
    "enough time to re-check my snowboard."
)
hotel_message = "Great, thanks. Can you recommend a good hotel near Delhi airport since I'm stuck here tonight?"
new_fare_message = "What is the baggage allowance for the new Business Lite fare?"

for message in [cancellation_message, hotel_message, new_fare_message]:
    print("Prasad:", message)
    print("FlyBot:", flybot_reply(message))
    print()

Each reply shows one of FlyBot's three limits.

**Adaptability.** FlyBot runs a fixed script, a flowchart of if-then-else rules. It spots the word "canceled" and sends a generic rebooking link. There is no rule for this combination (cancellation, snowstorm, missed start of a trip, oversized baggage), and FlyBot cannot reason that a snowstorm means no flights until it clears, or that Prasad is stressed and needs a complete answer rather than a link.

**Scope.** A hotel for a stranded passenger is the obvious next step of the same trip, but it is outside FlyBot's predefined world, so it refuses. It is like a skilled cashier who has no idea where the milk aisle is.

**Knowledge acquisition.** Its knowledge is hand-coded. When the airline launches a new Business Lite fare, developers must write new rules, design new conversation paths, test and deploy. Until then, FlyBot matches "baggage allowance" and confidently gives the old generic policy.

A modern agent approaches each differently. For the cancellation it recognises the real goal ("solve my disruption"), checks a weather service to confirm the warning, searches alternative flights, filters out any that leave too soon or allow too little time for the snowboard, and presents the best options for confirmation. For the hotel it sees a natural extension of the trip and calls a hotel search using the airport and arrival time. For Business Lite it reads the airline's current fare document at the moment of the question and answers from it, with no new code. You will build the first two in Chapter 4.

### 3.6 What changes when AI enters the picture

AI agents are the next step: not just agents, but agents supercharged with modern AI. Four shifts sum up the difference.

| From | To | Example |
|---|---|---|
| Narrow: specialists only | General and flexible: can work across domains | A chatbot that answers FAQs, summarises a document, recommends learning material and connects you to the right person |
| Rule-heavy: thousands of hand-written rules | Learning-powered: patterns learned from data | A self-driving system that learns from millions of road scenarios instead of a rule for every case |
| Rigid commands: `MOVE_FORWARD(10)` | Natural language: "move forward a little and stop near the lamp post" | Understands intent, tone and context; we no longer adapt to machines, machines adapt to us |
| Stuck in one box: its pre-coded world | Tool-using and resourceful: APIs, databases, other services | A travel agent that searches flights, compares prices, books, and updates your calendar |

You have already written code for three of these rows today: natural language understanding in 3.1, learning from feedback in 3.4, and planning in 3.3. The last row, tools, is the one that finally lets the concierge book the table.

### 3.7 Modern AI agents: the supercharged assistant

Today's agents combine every earlier ability, perception, reasoning, planning and learning, with language understanding and tool use. Like a fusion reactor, they merge several capabilities into one system. The table maps the six capabilities in the session document to where you meet each one in this notebook.

| Capability | What it means | Example | In this notebook |
|---|---|---|---|
| Perception | Understand meaning, intent and context in language, images and more | "Book me a flight to Delhi next week and remind me about my meeting" is recognised as two tasks | 3.1: structured understanding |
| Reasoning and planning | Break a goal into steps, order them, choose when to use a tool | "Plan a 3-day trip to Paris under $1000": split the budget, search flights and hotels, compare, build an itinerary | 3.3: planning and replanning |
| Memory | Keep context so you never repeat yourself | A research assistant remembers your topic and which sources it already searched | 3.2: the conversation list |
| Learning and adaptation | Adjust to preferences and changes without reprogramming | Spotify's AI DJ adapts to your taste (through recommendation models trained on listening data) | 3.4: lessons in memory |
| Tool use and action | Call APIs, run code, query databases, book tickets | A doctor's assistant reads patient history, queries a medical database, suggests diagnoses for the doctor, books a specialist | Chapter 4 |
| Collaboration and autonomy | Work with humans or other agents; keep going until the goal is done | AutoGPT, told "create a website about Italian recipes", plans, searches, writes code, tests and improves it | Chapter 4 and the project |

The step from Stage 5 to Stage 6 is easiest to see side by side. A Stage 5 agent searches flights from Delhi to London. A Stage 6 agent, told you need to be in London for meetings, also checks that your visa is still valid, looks at your calendar for clashes, and books a hotel near the meeting venue. It is context-aware and coordinates several actions intelligently, almost like a real assistant.

One caution about dates: the session document places modern agents from 2015. That fits deep reinforcement-learning agents like AlphaGo; agents that use a language model to choose tools became practical around 2022 (the ReAct paper appeared that October, AutoGPT in March 2023).

## 4. Tuesday: giving the concierge hands

Back to Monday's request. We now give Gemini the three things it lacked: tools (hands), a way to know today's date (eyes on the present), and a loop (a second look). The picture shows the whole plan. Keep it in view: every cell in this chapter is one arrow in it.

![The tool-calling loop](attachment:toolloop.png)

*The agent loop in code. Gemini only ever requests a tool; your Python code runs it and sends the result back.*

The key idea of this chapter, and of every agent framework you will meet: **the model decides the next step, and your code carries it out.**

### 4.1 Tools are ordinary Python functions

A tool is a normal function with three things the model can read: a clear name, type hints, and a docstring saying what it does and what each argument means. The model never sees the function body. The docstring is effectively a prompt, so write it carefully.

Prasad's calendar already has one event, an office cricket match this Saturday evening. Dates are computed from today so the example always works.

In [ ]:
from datetime import date, timedelta

today = date.today()
days_until_saturday = (5 - today.weekday()) % 7
if days_until_saturday == 0:
    days_until_saturday = 7
this_saturday = today + timedelta(days=days_until_saturday)
this_sunday = this_saturday + timedelta(days=1)

prasads_calendar = [
    {"date": this_saturday.isoformat(), "time": "18:00-21:00", "title": "Office cricket match at Shivaji Park"},
]

def get_today() -> str:
    """Return today's date and weekday. Use it before working out dates like 'this Saturday'."""
    now = date.today()
    return f"Today is {now.isoformat()} ({now.strftime('%A')})."

def check_calendar(date_text: str) -> str:
    """List the events in Prasad's calendar on one date.

    Args:
        date_text: The date in YYYY-MM-DD format, for example 2026-10-03.
    """
    found = []
    for event in prasads_calendar:
        if event["date"] == date_text:
            found.append(event["time"] + " " + event["title"])
    if len(found) == 0:
        return f"No events on {date_text}."
    return f"Events on {date_text}: " + "; ".join(found)

def add_calendar_event(date_text: str, time_text: str, title: str) -> str:
    """Add an event to Prasad's calendar.

    Args:
        date_text: The date in YYYY-MM-DD format, for example 2026-10-03.
        time_text: The start time in 24-hour HH:MM format, for example 19:00.
        title: A short description of the event.
    """
    # TODO: HW2 [Medium] - This tool accepts anything. If the model sends date_text="Saturday", the event is saved
    # with a date check_calendar can never match. What should the tool check, and what should it return so the
    # model notices and fixes its own mistake? (Think about the Feedback step of the agent loop.)
    prasads_calendar.append({"date": date_text, "time": time_text, "title": title})
    return f"Added: {date_text} {time_text} {title}"

# Run the tools ourselves first, before any model is involved
print(get_today())
print(check_calendar(this_saturday.isoformat()))
print(check_calendar(this_sunday.isoformat()))

### 4.2 What the model actually sees

`bind_tools` attaches tool descriptions to every call. LangChain builds each description from your function. Printing one shows exactly what Gemini reads: the name, the docstring's first line as the description, and the `Args:` lines as parameter descriptions.

In [ ]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(add_calendar_event), indent=2))

Notice what is not there: nothing says "check for clashes before adding" or "never invent a date". Rules like that exist only if you write them, in a docstring or in the system prompt.

### 4.3 One decision: the model asks, it does not act

**Before you run it:** will this single call change `prasads_calendar`?

In [ ]:
calendar_tools = [get_today, check_calendar, add_calendar_event]
concierge_with_tools = llm.bind_tools(calendar_tools)

concierge_instructions = (
    "You are DilSeVector's concierge. You can read and change Prasad's calendar with the tools provided. "
    "Always get today's date with a tool before working out a date like 'this Saturday'. "
    "Check the calendar for clashes before adding anything. You cannot book tables at the café itself; "
    "if asked, say so plainly and only add the calendar entry."
)
goal = "Book a table for two at the pink café in Dadar this Saturday at 7 pm and add it to my calendar."

messages = [SystemMessage(content=concierge_instructions), HumanMessage(content=goal)]
first_decision = concierge_with_tools.invoke(messages)

print("Text:", repr(first_decision.text))
print("Tool calls requested:")
for tool_call in first_decision.tool_calls:
    print(f"  {tool_call['name']}  args={tool_call['args']}  id={tool_call['id']}")
print("\nEvents in the calendar:", len(prasads_calendar))

The reply contains *tool calls*: structured requests such as "please run `get_today`". Nothing ran. The calendar still holds only the cricket match. This is the single most important fact about agents: **the model only requests; your code decides whether to act.** That is also where your control and safety live: you can refuse a request, check its arguments, or ask a human to confirm.

### 4.4 Act, and feed the result back

To complete one turn of the loop: keep the model's reply in the conversation, run each requested tool, wrap each result in a `ToolMessage` carrying the same `id` as the request, and ask the model again.

In [ ]:
tools_by_name = {"get_today": get_today, "check_calendar": check_calendar, "add_calendar_event": add_calendar_event}

# Step 1: keep the model's request in the conversation
messages.append(first_decision)

# Step 2: run each tool it asked for, and add the result
for tool_call in first_decision.tool_calls:
    tool_function = tools_by_name[tool_call["name"]]
    result = tool_function(**tool_call["args"])      # ** passes the dictionary as named arguments
    print(f"Ran {tool_call['name']} -> {result}")
    messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))

# Step 3: ask again, now that the model can see the results
second_decision = concierge_with_tools.invoke(messages)
print("\nNext request from the model:")
for tool_call in second_decision.tool_calls:
    print(f"  {tool_call['name']}  args={tool_call['args']}")
if len(second_decision.tool_calls) == 0:
    print("  (none) Final text:", second_decision.text)

Having seen today's date, the model typically asks to check Saturday next. Map this onto the loop from Chapter 2: the tool result is **perception**, the model's next request is **reasoning and planning**, running the tool is **action**, and appending the result is **feedback**.

### 4.5 The whole loop

Writing those three steps by hand for every turn would be tedious, so we put them in a `for` loop. It stops when the model replies without requesting a tool (it thinks the goal is done), or when it reaches `max_steps`. The step limit is not optional: a confused model can keep requesting tools until your daily quota runs out.

In [ ]:
def run_agent(goal, instructions, tools, max_steps=6):
    """The agent loop: let the model pick tools until it answers or runs out of steps."""
    tools_by_name = {}
    for tool in tools:
        tools_by_name[tool.__name__] = tool      # __name__ is the function's name as text, e.g. "get_today"
    model_with_tools = llm.bind_tools(tools)
    messages = [SystemMessage(content=instructions), HumanMessage(content=goal)]

    for step in range(1, max_steps + 1):
        reply = model_with_tools.invoke(messages)
        messages.append(reply)
        if len(reply.tool_calls) == 0:
            print(f"Step {step}: no tool requested, so the agent is done.")
            return reply.text, messages
        for tool_call in reply.tool_calls:
            # TODO: HW3 [Medium] - If the model asks for a tool that does not exist, or sends a wrong argument name,
            # the next two lines crash the whole agent. What should the loop send back instead, so the model can
            # try again? Test it by renaming a docstring argument.
            tool_function = tools_by_name[tool_call["name"]]
            result = str(tool_function(**tool_call["args"]))
            print(f"Step {step}: model chose {tool_call['name']}({tool_call['args']}) -> {result}")
            messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))

    return "Stopped: step limit reached without a final answer.", messages

# Reset the calendar so this run starts clean
prasads_calendar = [
    {"date": this_saturday.isoformat(), "time": "18:00-21:00", "title": "Office cricket match at Shivaji Park"},
]
print("Calendar reset:", prasads_calendar)

Two goals, same code, same tools. The first is a question. The second contains a condition that depends on what the calendar says.

**Before you run it:** how many tool calls will each goal need? Write your guess down.

In [ ]:
answer, trace = run_agent("Am I free this Saturday evening?", concierge_instructions, calendar_tools)
print("\nConcierge:", answer)

In [ ]:
goal_b = ("Book a table for two at the pink café in Dadar this Saturday at 7 pm and add it to my calendar. "
          "If I'm busy then, use Sunday at 7 pm instead.")

# TODO: HW1 [Easy] - After this runs, continue the conversation with "Actually, make it 8 pm." There is no tool to
# move or delete an event. Predict what the agent will do, test it, then write the missing tool.
answer_b, trace_b = run_agent(goal_b, concierge_instructions, calendar_tools)
print("\nConcierge:", answer_b)
print("\nPrasad's calendar now:")
for event in prasads_calendar:
    print("  ", event)

Compare with Monday. The calendar now has a new entry, most likely on Sunday at 19:00: the agent checked Saturday, saw the cricket match running until 21:00, and used the fallback. Nobody wrote an `if` statement about cricket. The model made that decision from a tool result.

Read the final answer critically too. Did it claim the café table is booked? It has no tool for that; our instructions told it to say so. Without that sentence in the system prompt, many runs would imply a booking that never happened. An agent is only as honest as its instructions and tools allow.

The conversation list is the agent's memory for this run. Every call resends all of it, so each step costs a little more than the last.

In [ ]:
for message in trace_b:
    kind = type(message).__name__
    if kind == "AIMessage" and len(message.tool_calls) > 0:
        names = []
        for tool_call in message.tool_calls:
            names.append(tool_call["name"])
        print(f"{kind:14s} requests {', '.join(names)}")
    else:
        print(f"{kind:14s} {message.text[:80]}")
print(f"\nThe last call resent all {len(trace_b)} messages.")

### 4.6 The short form: `create_agent`

You wrote the loop by hand so you know exactly what it does. In practice you will usually let LangChain run it. `create_agent` takes a model, a list of tools and a system prompt, and returns an agent with the same loop inside. This is how you will see agents written elsewhere, and it is built on LangGraph, which a later session covers in depth.

In [ ]:
from langchain.agents import create_agent

prasads_calendar = [
    {"date": this_saturday.isoformat(), "time": "18:00-21:00", "title": "Office cricket match at Shivaji Park"},
]

concierge_agent = create_agent(model=llm, tools=calendar_tools, system_prompt=concierge_instructions)

result = concierge_agent.invoke({"messages": [HumanMessage(content=goal_b)]})

for message in result["messages"]:
    kind = type(message).__name__
    if kind == "AIMessage" and len(message.tool_calls) > 0:
        for tool_call in message.tool_calls:
            print(f"{kind:14s} -> {tool_call['name']}({tool_call['args']})")
    else:
        print(f"{kind:14s} {message.text[:100]}")

print("\nCalendar:", prasads_calendar)

Same behaviour, a quarter of the code. The framework removed the plumbing; it did not remove any decisions. You still chose the tools, wrote the docstrings, and wrote the instructions that keep the agent honest.

### 4.7 FlyBot, rebuilt as an agent

Now the real test from Chapter 3.5. Prasad is stranded in Delhi. We give an agent four tools that stand in for the airline's systems. The data below is **pretend data written for this exercise**; in a real app each function would call the airline's, weather service's or hotel site's API.

In [ ]:
# ---- Pretend airline systems for this exercise ----
weather_alerts = {
    "KUU": "Snowstorm warning at Kullu-Manali (Bhuntar) airport until 10:00 tomorrow. No arrivals or departures before then.",
    "DEL": "No weather alerts.",
}
flights_delhi_to_kullu_tomorrow = [
    {"flight": "6E 2071", "departs": "07:15", "arrives": "08:40"},
    {"flight": "6E 2073", "departs": "11:05", "arrives": "12:30"},
    {"flight": "AI 9811", "departs": "13:20", "arrives": "14:45"},
]
baggage_policies = {
    "snowboard": "Snowboards are oversized baggage. Check them in at the special baggage counter at least 2 hours before departure.",
}
hotels_near_airports = {
    "DEL": "Aerocity Stay (2 km, Rs 6,500, free airport shuttle); Terminal Inn (4 km, Rs 3,900, shuttle every 30 min).",
}

def check_weather_alerts(airport_code: str) -> str:
    """Return current weather alerts for an airport.

    Args:
        airport_code: Three-letter airport code, for example KUU for Kullu or DEL for Delhi.
    """
    if airport_code in weather_alerts:
        return weather_alerts[airport_code]
    return "No weather alerts."

def search_flights(origin: str, destination: str) -> str:
    """Search tomorrow's flights between two airports.

    Args:
        origin: Three-letter code of the departure airport, for example DEL.
        destination: Three-letter code of the arrival airport, for example KUU.
    """
    if origin == "DEL" and destination == "KUU":
        lines = []
        for flight in flights_delhi_to_kullu_tomorrow:
            lines.append(f"{flight['flight']} departs {flight['departs']} arrives {flight['arrives']}")
        return "Tomorrow: " + "; ".join(lines)
    return "No flights found for that route in this demo."

def get_baggage_policy(item: str) -> str:
    """Return the airline's baggage rules for a special item.

    Args:
        item: The item, for example snowboard.
    """
    lowered = item.lower()
    for known_item in baggage_policies:
        if known_item in lowered:
            return baggage_policies[known_item]
    return "Standard baggage rules apply."

def search_hotels_near_airport(airport_code: str) -> str:
    """Find hotels close to an airport for a stranded passenger.

    Args:
        airport_code: Three-letter airport code, for example DEL.
    """
    if airport_code in hotels_near_airports:
        return hotels_near_airports[airport_code]
    return "No hotels found in this demo."

airline_tools = [check_weather_alerts, search_flights, get_baggage_policy, search_hotels_near_airport]
flybot_instructions = (
    "You are FlyBot 2.0, an airline assistant. Use the tools to check facts before you answer. "
    "Do not book anything yourself: propose the best option with your reasons and ask Prasad to confirm."
)
flybot_agent = create_agent(model=llm, tools=airline_tools, system_prompt=flybot_instructions)
print("FlyBot 2.0 has", len(airline_tools), "tools.")

**Before you run it:** three flights leave tomorrow. Which one should the agent recommend, and why? (Two facts matter, and they come from two different tools.)

In [ ]:
flybot_result = flybot_agent.invoke({"messages": [HumanMessage(content=cancellation_message)]})

for message in flybot_result["messages"]:
    if type(message).__name__ == "AIMessage" and len(message.tool_calls) > 0:
        for tool_call in message.tool_calls:
            print(f"-> {tool_call['name']}({tool_call['args']})")

print("\nFlyBot 2.0:", flybot_result["messages"][-1].text)

A good run checks the weather alert (nothing before 10:00, so 6E 2071 is out), searches flights, reads the snowboard rule (check in two hours early), and recommends 6E 2073 at 11:05, asking Prasad to confirm. That is **adaptability**: no rule was written for "snowstorm plus snowboard"; the agent combined tool results to fit the situation.

Now the follow-up that old FlyBot refused. We pass the whole conversation back, plus the new message, so the agent remembers the context.

In [ ]:
follow_up_messages = flybot_result["messages"] + [HumanMessage(content=hotel_message)]
hotel_result = flybot_agent.invoke({"messages": follow_up_messages})

for message in hotel_result["messages"][len(follow_up_messages):]:
    if type(message).__name__ == "AIMessage" and len(message.tool_calls) > 0:
        for tool_call in message.tool_calls:
            print(f"-> {tool_call['name']}({tool_call['args']})")

print("\nFlyBot 2.0:", hotel_result["messages"][-1].text)

That is **scope**: the hotel is part of the same journey, and the agent had a tool for it. It may even pick the hotel with the shuttle because of the early check-in. Remember where that flexibility came from: someone built each tool. An agent cannot reach beyond the tools it is given.

## 5. Why agents, and why not always

We have seen what agents can do, so why not use them for everything?

Real life is not single questions and single answers. Plan a weekend trip with only a chat model and you get pieces of the puzzle, a hotel idea here and a flight there, which you stitch together yourself. An agent can observe the context, plan the steps, act with several tools, remember what it has done, and adapt when something changes. That is why agents are becoming essential: they bridge human intent and real-world execution.

But that flexibility has costs.

**Calls and time.** "Am I free on Saturday?" took three model calls. A calendar app answers it with none.

**Predictability.** Run goal B several times and the traces differ: sometimes both days are checked at once, sometimes one at a time. You cannot write a test that expects an exact sequence.

**Reliability compounds.** If each step is right 95% of the time, a run of several steps is right only if *every* step is:

| Steps in a run | Chance all steps are right (0.95 per step) |
|---|---|
| 1 | 95% |
| 3 | 86% |
| 5 | 77% |
| 10 | 60% |

A 10-step agent built from excellent 95% steps fails four times in ten. So before reaching for an agent, ask the question the session document asks: **do we always need an agent for every task?** No. Sometimes the problem is predictable and can be solved by a series of predefined steps. That is a **workflow**.

### What a workflow is

A workflow is a predefined series of steps executed in a set order to reach an outcome, like a business process: submit an expense report, manager approves, finance approves, reimbursement. The sequence is known in advance and the system just follows it.

Workflows matter in AI development for three reasons.

**The process is predictable.** When the steps never change, improvisation only adds risk. A CI/CD pipeline always runs tests, builds and deploys; you would never want it to decide whether to test today. A document ingestion pipeline always runs OCR, text cleaning, LLM classification, then storage.

**Reliability matters more than creativity.** Payment processing must validate the card, process payment, update the ledger and send a receipt, every time. An e-commerce order must take payment, update inventory, generate an invoice, trigger shipping and notify the customer. Skipping one step causes real damage.

**Efficiency is key.** A nightly ETL job extracts, transforms and loads without "thinking" about what to do. A support workflow classifies the ticket, routes it to the right team, drafts a first response and escalates if unresolved, quickly and at scale.

### Workflows or agents?

Both are strategies for tasks an LLM cannot complete in a single request, and you have built both in this course. When you gave the model tools and let it decide how to complete a task (Chapter 4), that was an agent. When you chained fixed steps in Session 6 or chose a retrieval strategy in Session 5's adaptive RAG, those were workflows.

- Use a **workflow** when you can picture the exact steps the LLM should go through, or when your app's interface limits users to a set of tasks. A workflow is a series of LLM calls through predetermined steps.
- Use an **agent** when you cannot know in advance what task, or what task parameters, you will be given. An agent gets a goal and a set of tools and works out how to use them.

The session document's example is a web app that turns a photo of a metal part into a 3D CAD file (a STEP file). You know exactly what to do with every uploaded image, so it is a workflow:

1. Feed the image to the LLM and ask it to describe the object.
2. Ask the LLM to model the object with the CadQuery library, based on that description.
3. Render the model.
4. Ask the LLM to grade the rendering against the original image, and fix any problems.

Workflow patterns are battle-tested recipes that engineers have used again and again. Learning them does not implement anything for you; you still write the code and adapt it. But they show you a path that works. Here are the four the session document covers, in its order.

## 6. Wednesday: four workflow patterns

### 6.1 Evaluator-optimizer

The CAD workflow's last step is an **evaluator-optimizer** pattern.

![Evaluator-optimizer](attachment:evaluator_optimizer.png)

*A producer creates a submission; a grader accepts it or sends feedback; the loop repeats until accepted.*

**Producer:** takes the input and creates output (the LLM writing CadQuery code and rendering it). **Grader:** evaluates the output against criteria. **Feedback loop:** if the grader rejects it, its feedback goes back to the producer. **Iteration:** repeat until the grader accepts.

We cannot run the CAD example here, so we use a job DilSeVector needs every day: writing profile bios. Prasad wants a new one for himself. A producer LLM writes it; a grader LLM checks it against clear criteria and returns structured feedback; our code loops until it passes or three rounds are used.

In [ ]:
bio_criteria = """1. Under 50 words.
2. Mentions coffee and, playfully, vector search.
3. No cliches such as 'partner in crime', 'love to laugh' or 'work hard, play hard'.
4. Ends with a question that invites a reply."""

grade_schema = {
    "title": "grade",
    "description": "Whether a dating-app bio meets the criteria",
    "type": "object",
    "properties": {
        "accepted": {"type": "boolean"},
        "feedback": {"type": "string", "description": "Which criteria failed and how to fix them. Empty if accepted."},
    },
    "required": ["accepted", "feedback"],
}
grader_llm = llm.with_structured_output(grade_schema)

producer_prompt = ("Write a dating-app bio for Prasad, 29, from Dadar, who built DilSeVector and loves kesar cold coffee. "
                   "Return only the bio.")

bio = llm.invoke(producer_prompt).text          # producer, round 1
for round_number in range(1, 4):
    grade = grader_llm.invoke(f"Criteria:\n{bio_criteria}\n\nBio:\n{bio}\n\nGrade the bio strictly against every criterion.")
    print(f"Round {round_number}: {len(bio.split())} words, accepted = {grade['accepted']}")
    print("  Bio:", bio)
    if grade["accepted"]:
        break
    print("  Grader feedback:", grade["feedback"])
    bio = llm.invoke(f"{producer_prompt}\n\nYour previous bio:\n{bio}\n\nAn editor said:\n{grade['feedback']}\n\nFix these problems.").text

Watch the feedback: the grader names what failed, and the next draft fixes that. The loop can stop after one round (a good first draft costs two calls) or use all three. The round limit is essential, just like the agent's step limit.

Its weakness is the grader. A grader is another LLM call with its own mistakes; if producer and grader are the same model, they can share blind spots. Where a criterion is mechanical (a word count), plain code is a cheaper and more reliable grader.

### 6.2 Parallelization

Imagine a materials app: users upload an image of a part and receive the best material for it. Your first instinct is one prompt listing metal, polymer, ceramic, composite, elastomer and wood, with detailed criteria for each. The session document shows that prompt; expand it below. It works, but the model must juggle dozens of competing criteria at once, and results are vague.

<details>
<summary>The single, complex prompt (click to expand)</summary>

> You are an expert Material Science Assistant helping engineers select the most suitable material for a given part. A user will upload an image of a part. Your task is to analyze the image and recommend the best material from: Metal, Polymer, Ceramic, Composite, Elastomer, Wood. Use the following material-specific criteria: **Metal**: high strength and stiffness, durability, electrical conductivity, thermal conductivity, high operating temperature, machinability, magnetic properties, recyclability. **Polymer**: lightweight, corrosion resistance, flexibility, insulating properties, low cost and high volume, impact resistance, aesthetics. **Ceramic**: high hardness, high temperature resistance, chemical inertness, electrical insulation, low ductility acceptable, dimensional stability. **Composite**: high strength-to-weight ratio, tailored properties, corrosion resistance, thermal or acoustic insulation, complex shapes, cost versus performance. **Elastomer**: flexibility and elasticity, impact absorption, sealing and waterproofing, low temperature performance, chemical resistance, durability under repeated deformation. **Wood**: aesthetic appearance, moderate strength, lightweight, machinability, thermal insulation, cost effectiveness, sustainability. Recommend one primary material, one or two alternatives, and the key limitation. Output JSON with recommended_material, reasoning, alternative_options and limitations.

</details>

A better approach is to split the task. Send the part to the LLM several times at once, each time with the criteria for **one** material, then send all the analyses to a final call that compares them and decides.

![Parallelization for material selection](attachment:parallel_materials.png)

*Each material gets its own focused evaluation; a final call aggregates. The picture shows Claude in each box; any LLM fits there, and we use Gemini.*

The general shape: split one task into sub-tasks, run them in parallel, aggregate. The sub-tasks need not be identical; each can have its own prompt, tools or criteria.

![The parallelization pattern](attachment:parallel_structure.png)

*The parallelization pattern in general.*

The pink café's coffee grinder has a worn burr, the part that crushes the beans, and the owner asks Prasad which material the replacement should be. We evaluate four materials, as in the picture, with the criteria from the session document. `llm.batch` sends all four prompts together.

In [ ]:
part_description = ("A burr for a commercial coffee grinder in a busy café: a toothed disc about 64 mm across that "
                    "crushes roasted beans all day, must stay sharp, must be food-safe, and must not warm the coffee.")

material_criteria = {
    "Metal": "high strength and stiffness, durability against wear and fatigue, thermal conductivity, machinability, recyclability",
    "Polymer": "lightweight, corrosion resistance, flexibility, low cost for mass production, impact resistance",
    "Ceramic": "high hardness and wear resistance, high temperature resistance, chemical inertness, dimensional stability, low ductility acceptable",
    "Composite": "high strength-to-weight ratio, tailored properties, corrosion resistance, complex shapes, cost versus performance",
}

material_names = []
material_prompts = []
for material in material_criteria:
    material_names.append(material)
    material_prompts.append(
        f"You are a materials engineer. Judge ONLY whether {material} suits this part, using these criteria: "
        f"{material_criteria[material]}.\nPart: {part_description}\n"
        "Give a suitability score from 1 to 10 and a reason in under 60 words."
    )

material_replies = llm.batch(material_prompts)       # four focused evaluations, sent together

analyses_text = ""
for position in range(len(material_names)):
    name = material_names[position]
    print(f"=== {name} ===")
    print(material_replies[position].text)
    print()
    analyses_text = analyses_text + f"<{name}>\n{material_replies[position].text}\n</{name}>\n"

In [ ]:
aggregator_prompt = (
    f"Four specialists each evaluated one material for this part:\n{part_description}\n\n{analyses_text}\n"
    "Compare their analyses. Recommend one primary material, one alternative, and the key limitation of "
    "the primary choice, in under 80 words."
)
recommendation = llm.invoke(aggregator_prompt)
print(recommendation.text)

Most runs recommend metal (hardened steel) or ceramic, with the other as the alternative, which is what real grinders use. The benefits of the split, from the session document: **focused attention** (each call weighs one material's criteria properly), **easier optimisation** (if the metal analysis is weak, fix only that prompt), **scalability** (adding elastomer and wood means adding two prompts, nothing else) and **reliability** (less juggling, more consistent results).

One honest note: on the free tier, the rate limiter lets only one call through every 15 seconds, so the four "parallel" calls still take about a minute. Parallelization gave us focus. On a paid tier it would also give speed.

Use it when a decision splits into independent judgements. If the sub-tasks depend on each other, you want a chain instead.

### 6.3 Chaining

A **chaining** workflow breaks a large task into sequential subtasks, each building on the last.

![Chaining](attachment:chaining.png)

*Each processing task takes the previous one's output.*

The session document's example is a social media tool that creates and posts videos: find trending topics on Twitter, select the most interesting (LLM), research it (LLM), write a short video script (LLM), create the video with an AI avatar and text-to-speech, post it. Why not one giant prompt? Because a model given one job at a time does each job better. Chaining splits large tasks that cannot run in parallel, lets you run ordinary code between steps, and keeps each LLM call focused.

It also solves **the long prompt problem**. Ask an LLM for a technical article with a list of constraints (don't mention being an AI, no emojis, no cliches, professional tone) and even when every constraint is clear, some drafts break a few. Instead of fighting one huge prompt, accept an imperfect first draft, then send a second, focused request that only revises.

Prasad wants a post for DilSeVector's tech blog. These are the session document's prompts, with a word limit added so the output fits the notebook.

In [ ]:
article_subject = "Prompt Engineering for AI Language Models"

article_prompt = f"""Write a comprehensive, high-quality article on a particular programming technique, suitable for publication on a reputable technical blog or documentation platform.

The programming technique you are required to write about is:
<article_subject>
{article_subject}
</article_subject>

CONSTRAINTS (ABSOLUTELY DO NOT DO THE FOLLOWING):
- DO NOT simply define the term and stop. The article must be comprehensive.
- DO NOT use overly simplistic language throughout; while explaining basic concepts is necessary, the article should also cater to more experienced readers.
- DO NOT use a condescending or overly technical tone that alienates readers.
- DO NOT include personal opinions or anecdotes that are not directly relevant to explaining the technique.
- DO NOT mention or discuss the fact that you are an AI or a language model. Write as if you are a human expert technical writer.
- DO NOT use emojis!
- DO NOT use cliched or cringey writing!

Keep it under 350 words."""

# Step 1: write, accepting that the first draft may break some rules
draft = llm.invoke(article_prompt).text

# Step 2: a focused revision request
revision_prompt = f"""Revise the article provided below. Follow these steps to rewrite the article:
1. Identify any location where the text identifies the author as an AI and remove them
2. Find and remove all emojis
3. Locate any cringey writing and replace it with text that would be written by a technical writer
Return only the revised article.

<article>
{draft}
</article>"""
revised = llm.invoke(revision_prompt).text

print("=== Draft (first 600 characters) ===")
print(draft[:600])
print("\n=== Revised (first 600 characters) ===")
print(revised[:600])

Compare the two openings: phrases like "In today's fast-paced world" or "Let's dive in" tend to disappear in the revision, because the second call's only job is to hunt for them.

That was the long form, two named steps. You will more often see chains written in LangChain's short form, the pipe syntax from Session 6, which builds the same two calls:

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

write_chain = ChatPromptTemplate.from_template(article_template) | llm | StrOutputParser()
revise_chain = ChatPromptTemplate.from_template(revision_template) | llm | StrOutputParser()

revised = revise_chain.invoke({"article": write_chain.invoke({"article_subject": article_subject})})
```

The session document's version uses `LLMChain` and `SequentialChain`. Those were removed in LangChain 1.x, and `from langchain.chains import LLMChain` now fails. If you meet that code in older tutorials, translate it into one of the forms above.

Use chaining when a task has several requirements, when the model keeps ignoring some constraints in a long prompt, when you need to check or process output between steps, or when you want each call small and testable.

### 6.4 Routing

Different requests need different handling. DilSeVector's marketing team (Prasad) makes short videos. "Python functions" and "surfing" should not get the same script: programming calls for a clear educational explanation, surfing for energy and visuals. One generic prompt cannot do both well.

A **routing** workflow first categorises the input, then sends it to one specialised pipeline.

![Routing](attachment:routing.png)

*The router sends each input to exactly one specialised path.*

The session document defines six genres, each with its own prompt template:

| Genre | Script prompt |
|---|---|
| Entertainment | Write a high-energy, culturally-relevant script about [TOPIC] using trendy language and engaging hooks that balance entertainment value with insider insights. |
| Educational | Develop a clear, engaging script about [TOPIC] that transforms complex information into digestible insights using relatable examples and thought-provoking questions. |
| Comedy | Create a sharp, unexpected script about [TOPIC] that subverts expectations with clever observations, perfect timing, and relatable scenarios that build to satisfying punchlines. |
| Personal vlog | Write an authentic, intimate script about [TOPIC] that balances vulnerability with relatable insights, using conversational storytelling that invites viewers into your personal experience. |
| Reviews | Create a decisive, experience-based script about [TOPIC] highlighting key strengths and weaknesses with specific examples that support your clear final verdict. |
| Storytelling | Write an immersive script about [TOPIC] using vivid details and strategic pacing that builds emotional connection through authentic storytelling. |

Routing happens in two steps: **categorisation** (an LLM call assigns a genre) and **specialised processing** (the matching template generates the script). We use structured output with an `enum`, so the router can only answer with one of our six genres.

**Before you run it:** which genre will each of Prasad's topics get?

In [ ]:
genre_prompts = {
    "Entertainment": "Write a high-energy, culturally-relevant script about [TOPIC] using trendy language and engaging hooks that balance entertainment value with insider insights.",
    "Educational": "Develop a clear, engaging script about [TOPIC] that transforms complex information into digestible insights using relatable examples and thought-provoking questions.",
    "Comedy": "Create a sharp, unexpected script about [TOPIC] that subverts expectations with clever observations, perfect timing, and relatable scenarios that build to satisfying punchlines.",
    "Personal vlog": "Write an authentic, intimate script about [TOPIC] that balances vulnerability with relatable insights, using conversational storytelling that invites viewers into your personal experience.",
    "Reviews": "Create a decisive, experience-based script about [TOPIC] highlighting key strengths and weaknesses with specific examples that support your clear final verdict.",
    "Storytelling": "Write an immersive script about [TOPIC] using vivid details and strategic pacing that builds emotional connection through authentic storytelling.",
}

genre_names = []
for genre in genre_prompts:
    genre_names.append(genre)

router_schema = {
    "title": "route",
    "description": "The genre of a short video topic",
    "type": "object",
    "properties": {"genre": {"type": "string", "enum": genre_names}},
    "required": ["genre"],
}
router_llm = llm.with_structured_output(router_schema)

video_topics = [
    "What an embedding is, explained in 60 seconds",
    "How I met my partner at a pink café in Dadar, thanks to vector search",
]

for topic in video_topics:
    # Step 1: categorise
    route = router_llm.invoke(f"Categorize the topic of a video into one of the listed categories.\n<topic>{topic}</topic>")
    genre = route["genre"]
    # Step 2: specialised processing
    script_prompt = genre_prompts[genre].replace("[TOPIC]", topic) + " Keep it under 100 words."
    script = llm.invoke(script_prompt).text
    print(f"Topic: {topic}\nRouted to: {genre}\n{script}\n" + "-" * 60)

The embedding topic goes to Educational; the café story usually goes to Personal vlog or Storytelling. The key insight is that each input goes to **only one** pipeline, so every pipeline can be tuned for its own kind of input.

Routing works well when requests come in clearly different kinds, when you can define categories that cover them, when the LLM can categorise reliably, and when specialised handling is worth the extra call. It is common in customer service bots and content tools.

### 6.5 The patterns side by side

| Pattern | Shape | Calls | Use it when | Avoid it when |
|---|---|---|---|---|
| Evaluator-optimizer | Produce, grade, feed back, repeat | 2 up to 2 × rounds | You can state clear acceptance criteria | You cannot say what "good" means |
| Parallelization | Split, run independently, aggregate | N + 1 | One decision splits into independent judgements | Sub-tasks depend on each other |
| Chaining | Step 1 → step 2 → step 3 | One per step | Several dependent parts, or constraints a long prompt keeps dropping | One prompt already does the job |
| Routing | Classify, then one specialised path | 1 + the path | Inputs come in clearly different kinds | Categories overlap and the router guesses |

In all four, *your code* decided the order of steps before any request arrived. That is what makes them workflows.

## 7. Workflow or agent?

![Workflow versus agent](attachment:wf_vs_agent.png)

*In a workflow your code fixes the order. In an agent the model picks the next action.*

Think of a workflow as a **recipe**: you know the ingredients and the steps, and someone follows them carefully. Writing a blog post might be outline, draft, polish; analysing customer feedback might be categorise, extract sentiment, summarise. Every step is predictable, testable and reliable.

Think of an agent as arriving in a **new city** with a phone, some cash and a few apps. There is no plan, but you will find food, navigate, maybe book a cab. Given tools like a calculator, a search API or a booking system, an agent forms its own plan, mixes tools in unexpected ways, asks clarifying questions, and adapts to situations nobody prepared it for.

| | Workflows | Agents |
|---|---|---|
| Benefits | The LLM focuses on one subtask at a time, so accuracy is higher; easy to test and evaluate because you know each step; predictable and reliable; suited to specific, well-defined problems | Flexible user experience; combine tools in unexpected ways to complete a wide variety of tasks; handle situations nobody anticipated; can ask the user for more input |
| Downsides | Much less flexible, dedicated to specific kinds of task; constrained user experience, since inputs must fit the flow; more up-front planning and design | Lower task-completion rate; harder to instrument, test and evaluate, because you do not know which steps will run; less predictable |

### Tools for building them

**No-code tools.** Platforms such as **n8n** and Langflow let you build automations by connecting boxes in a visual editor. n8n is open source and self-hostable (so your data stays yours), has hundreds of ready integrations, and supports both drag-and-drop and custom code. Teams use it for lead management, turning support emails into tickets, DevOps alerts, reporting pipelines and e-commerce notifications. **This course does not teach or use no-code tools.** We build agents in code, with LangGraph and CrewAI in later sessions, because code gives you full control, flexibility and the ability to fix things when they break.

**Frameworks.** Many frameworks help build agent applications. You should know their names; the course covers the major ones.

| Framework | What it is known for |
|---|---|
| LangChain | Building blocks for LLM apps with tools, memory and reasoning; `create_agent` from Chapter 4 |
| LangGraph | Graph-based, stateful control of single or multi-agent systems; integrates with LangChain; covered later |
| CrewAI | Teams of role-based agents collaborating; covered later |
| AutoGen | Microsoft's open-source framework for collaborative multi-agent systems |
| LlamaIndex | Retrieval-first framework (Session 7) that brings your data into agent reasoning |
| OpenAI Agents SDK | Orchestrating teams of agents on OpenAI's platform |
| Smolagents | Hugging Face's lightweight library where agents act by writing code |
| Atomic Agents | A library for modular, composable agents |

## 8. Friday: planning the first trip

Prasad's flight is sorted. Now he and his partner want a day-by-day plan for Manali that they can load straight into their calendars. We build it three ways, just as the session document's demonstrations do, and compare them: **GenAI** (one prompt), an **AI agent** (one agent with a search tool), and the **agentic planner** (four specialised roles in sequence).

![The travel planner](attachment:planner.png)

*The four-role planner. Only the researcher is an agent that decides anything; the other steps always run in the same order.*

In [ ]:
destination = "Manali"
number_of_days = 3
preferences = "We are vegetarian, we prefer quiet places to crowded ones, and we like short walks."
trip_request = f"Create a {number_of_days}-day itinerary for {destination}. Preferences: {preferences}"
print(trip_request)

### 8.1 GenAI: one prompt

This is the demo the session document suggests running in Google AI Studio: ask for the itinerary directly. The model writes from what it learned during training.

In [ ]:
genai_itinerary = llm.invoke(trip_request + " Use the format 'Day 1: ...', 'Day 2: ...'.").text
print(genai_itinerary)

Fluent and plausible. But nothing was looked up: whether a café still exists, whether a road is open in December, whether a place is really quiet. It all comes from training data with a cut-off date, and the model cannot tell you which details are stale.

### 8.2 A search tool the agent can use

The session document's version used SerpAPI, which needs a separate paid-tier account. We use **Wikivoyage** instead, a free, open travel guide whose API needs no key. Its pages have standard sections (Understand, Get in, Get around, See, Do, Eat, Sleep, Stay safe), which suits a travel researcher. It is one source, not the whole web, and has no live prices. We run the tool ourselves first.

In [ ]:
import requests

def search_travel_guide(place: str) -> str:
    """Look up a town, city or region in the Wikivoyage travel guide and return its main sections
    (Understand, Get in, Get around, See, Do, Eat, Sleep, Stay safe).

    Args:
        place: The name of the place, for example Manali, Old Manali or Solang Valley.
    """
    response = requests.get(
        "https://en.wikivoyage.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1, "redirects": 1, "titles": place, "format": "json"},
        headers={"User-Agent": "GenAI-course-agents-notebook/1.0 (educational use)"},
        timeout=30,
    )
    pages = response.json()["query"]["pages"]
    page_text = ""
    for page_id in pages:
        if "extract" in pages[page_id]:
            page_text = pages[page_id]["extract"]
    if page_text == "":
        return f"No Wikivoyage page found for '{place}'. Try another spelling or a larger nearby place."

    wanted_sections = ["Understand", "Get in", "Get around", "See", "Do", "Eat", "Sleep", "Stay safe"]
    result = f"Wikivoyage guide for {place}:\n"
    current_section = ""
    for line in page_text.split("\n"):
        if line.startswith("== ") and line.endswith(" =="):
            current_section = line[3:-3].strip()
            if current_section in wanted_sections:
                result = result + f"\n## {current_section}\n"
        elif current_section in wanted_sections and len(result) < 8000:
            result = result + line + "\n"
    return result

guide_text = search_travel_guide(destination)
print(guide_text[:1500])

### 8.3 An AI agent: one agent with a tool

This is the single-agent version the session document calls `AI_travel.py`. One agent, one tool, and the model decides what to look up (the destination, perhaps a nearby valley it read about) and when it has enough to write the plan. You already know how to build it in a few lines with `create_agent`.

In [ ]:
single_agent_instructions = (
    "You are a travel planning agent. Use search_travel_guide to research the destination, and any nearby place "
    "the guide mentions that you want to include. Then write a day-by-day itinerary that uses only places and facts "
    "you found with the tool. If the guide does not say something, do not state it.\n"
    "Format:\nFinal Answer:\nDay 1: ...\nDay 2: ..."
)
travel_agent = create_agent(model=llm, tools=[search_travel_guide], system_prompt=single_agent_instructions)

travel_result = travel_agent.invoke({"messages": [HumanMessage(content=trip_request)]})
for message in travel_result["messages"]:
    if type(message).__name__ == "AIMessage" and len(message.tool_calls) > 0:
        for tool_call in message.tool_calls:
            print(f"-> {tool_call['name']}({tool_call['args']})")

single_agent_itinerary = travel_result["messages"][-1].text
print("\n" + single_agent_itinerary)

The session document lists this single agent's limits: it researches and plans everything itself, so results can be shallow; it can still hallucinate; nothing refines the plan; and nothing checks it. The agentic version answers each limit with a specialist role.

### 8.4 The agentic planner: four roles in sequence

Before building on the planner from the course materials (`agentic_travel_planner.py`), read its core. Two lines matter:

```python
def run(self, message, **kwargs):
    ...                                   # copies each tool's name and description into the prompt
    return self.model.predict(full_prompt)
```

**It no longer runs.** `.predict()` was removed from LangChain; on current versions this line raises `AttributeError`. The modern call is `.invoke()`.

**Its search tool is never used.** The tool's name and description are pasted into the prompt, but no code ever runs the search function. The researcher is *told* it has a search tool, cannot use one, and writes from memory. Remember Chapter 4.3: the model can only *request* a tool; your code has to run it. A tool that is described but never executed is the most common bug in hand-built agents, because everything still looks like it works.

The version below keeps the original's four roles and their instructions, with both problems fixed: the researcher is a real agent built with `create_agent` and the Wikivoyage tool, and every call uses `.invoke()`. Each role is a dictionary: a name, instructions, and tools.

In [ ]:
researcher = {
    "name": "Researcher Agent",
    "tools": [search_travel_guide],
    "instructions": """You are the Researcher Agent for a travel planning system.
Your goal is to collect reliable, up-to-date travel information for the user's trip.
You have access to the search_travel_guide tool, which reads the Wikivoyage travel guide.
Instructions:
1. Use the search tool where necessary (do not invent information).
2. Base every item on what the tool returned. If the guide does not cover something (for example hotel prices), say so.
3. Summarize findings in a structured way: Attractions (with highlights), Hotels/Accommodations, Restaurants/Food options, Travel/Transport tips.
4. Keep descriptions concise but informative (2-4 sentences per item).
5. Avoid adding your own opinions; base all output on retrieved sources.
Output must be a clear research summary. Do NOT create an itinerary.""",
}

planner = {
    "name": "Planner Agent",
    "tools": [],
    "instructions": """You are the Planner Agent. Your responsibility is to transform research results into a clear travel plan.
Instructions:
1. Create a day-by-day itinerary for the trip.
2. Each day should include a balance of attractions, food options, and rest time.
3. Ensure logical sequencing (minimize travel time, group nearby spots together).
4. Respect the number of days requested by the user.
5. Do not invent new information - only use the research provided.
6. Ensure each day has a clear theme or highlight (morning, afternoon, evening activities).
7. Write in an engaging but professional tone.
Wrap the final response in this format:
Final Answer:
Day 1: ...
Day 2: ...
Only provide the itinerary, nothing else.""",
}

optimizer = {
    "name": "Optimizer Agent",
    "tools": [],
    "instructions": """You are the Optimizer Agent. Your task is to refine the draft itinerary for maximum usability.
Instructions:
1. Improve the itinerary for efficiency (minimize unnecessary travel).
2. Adjust based on user preferences (e.g., prefers museums over nightlife).
3. Ensure realistic timings (no overpacked days, allow travel/rest time).
4. Add small enhancements (hidden gems, local experiences) where appropriate.
5. Keep it feasible and enjoyable for a real traveler.
Output the refined itinerary in the same day-by-day format, starting with 'Final Answer:'.""",
}

qa_checker = {
    "name": "QA Agent",
    "tools": [],
    # TODO: HW5 [Medium] - The QA agent is told to "remove hallucinations", but it only ever sees the itinerary.
    # How could it tell a real place from an invented one? What would you pass to it? (Session 5: corrective RAG;
    # Session 8: faithfulness.)
    "instructions": """You are the Quality Assurance Agent for travel itineraries.
Your responsibility is to validate and correct the final plan before presenting it to the user.
Instructions:
1. Check factual accuracy: remove hallucinations, unrealistic claims, or non-existent places.
2. Verify consistency: ensure attractions match the destination and sequence makes sense.
3. Ensure clarity: rewrite vague items (e.g., 'visit museum' -> 'Visit the Louvre Museum').
4. Confirm that each day is feasible (not overloaded, includes travel time).
5. Keep output user-friendly and safe (avoid unsafe areas or misleading advice).
Output the final verified itinerary in the same format, starting with 'Final Answer:'.""",
}

def run_role(role, message):
    """Roles with tools run as agents; roles without tools make one LLM call."""
    print(f"\n=== {role['name']} ===")
    if len(role["tools"]) > 0:
        role_agent = create_agent(model=llm, tools=role["tools"], system_prompt=role["instructions"])
        result = role_agent.invoke({"messages": [HumanMessage(content=message)]})
        output = result["messages"][-1].text
    else:
        output = llm.invoke([SystemMessage(content=role["instructions"]), HumanMessage(content=message)]).text
    print(output[:700] + "\n[...]")
    return output

print("Four roles ready.")

The orchestrator runs the four roles in a fixed order, exactly as in the original.

**Before you run it:** the preferences say "vegetarian". Which role is the first to find out?

In [ ]:
def agentic_travel_planner(destination, num_days, preferences=""):
    research = run_role(researcher, f"Research {destination} for a {num_days}-day trip. Include attractions, hotels, restaurants.")

    # TODO: HW4 [Easy] - The planner never sees the preferences, so it drafts a plan the optimizer must rework.
    # Which role should see them first, and what changes when it does?
    draft = run_role(planner, f"Create a {num_days}-day itinerary from this research:\n{research}")

    optimized = run_role(optimizer, f"Optimize this itinerary for user preferences: {preferences}\n{draft}")

    # TODO: HW6 [Hard] - QA runs once and its output is always accepted. Turn this step into an evaluator-optimizer
    # (Chapter 6.1): QA returns structured {accepted, feedback}, and a rejected plan goes back to the optimizer.
    # What round limit makes sense on a free tier?
    verified = run_role(qa_checker, f"Verify and correct this itinerary:\n{optimized}")

    return {"research": research, "draft": draft, "optimized": optimized, "verified": verified}

planner_outputs = agentic_travel_planner(destination, number_of_days, preferences)
print("\n=== FINAL ===\n" + planner_outputs["verified"])

Now ask this session's question: who decided each step? The orchestrator did, in a fixed order: research, plan, optimise, verify. Only the researcher made choices (which pages to read). So by the definitions in Chapter 5, this "agentic" planner is a **chaining workflow whose first step is an agent**. That is a good design, and a very common one: let a model decide where the path cannot be predicted, and keep fixed steps where it can. Just do not believe that a role called "QA" guarantees quality. It is a prompt, and it is only as good as the information it receives.

### 8.5 Did the itinerary come from the research?

The three itineraries all read well, so reading them will not tell you which to trust. An LLM can do a first check for us: give it the research and an itinerary, and ask it to list the places the itinerary mentions and whether each one appears in the research. This is **LLM-as-judge**, the same idea as the RAGAS faithfulness check from Session 8.

In [ ]:
grounding_schema = {
    "title": "grounding_check",
    "description": "Which places in an itinerary appear in the research",
    "type": "object",
    "properties": {
        "found_in_research": {"type": "array", "items": {"type": "string"}},
        "not_in_research": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["found_in_research", "not_in_research"],
}
judge_llm = llm.with_structured_output(grounding_schema)

itineraries = {
    "GenAI (no tool)": genai_itinerary,
    "Agentic planner": planner_outputs["verified"],
}
for label in itineraries:
    verdict = judge_llm.invoke(
        "List every named place, restaurant or attraction in the ITINERARY. Put each under found_in_research if the "
        "RESEARCH mentions it, otherwise under not_in_research.\n\n"
        f"RESEARCH:\n{guide_text}\n\nITINERARY:\n{itineraries[label]}"
    )
    print(f"{label}: {len(verdict['found_in_research'])} places found in the guide, {len(verdict['not_in_research'])} not found")
    print("   not found:", verdict["not_in_research"])

Expect the GenAI plan to have more places the guide never mentions; it never saw the guide. If the agentic planner has some too, look at the optimizer's instruction "add small enhancements (hidden gems, local experiences)": it asks a role with no tool to add things, and a role with no tool can only add them from memory. "Not in the research" does not mean false; it means unverified. The judge is itself an LLM and can be wrong, so treat its list as a place to start checking.

### 8.6 Into the calendar

The original app exports the final plan as an `.ics` file, which Google Calendar, Outlook and Apple Calendar all import. This is the original function, unchanged apart from homework comments.

In [ ]:
from icalendar import Calendar, Event
from datetime import datetime
import re

def generate_ics_content(plan_text: str, start_date: datetime = None) -> bytes:
    cal = Calendar()
    cal.add('prodid', '-Agentic FlyTripper-')
    cal.add('version', '2.0')
    if start_date is None:
        start_date = datetime.today()
    plan_text_without_final_answer = plan_text.replace("Final Answer:", "").strip()
    # TODO: HW7 [Easy] - If one day's text mentions another day ("save energy for Day 3's trek"), the calendar ends up
    # with the wrong number of events. Count the events against the days in the plan. Why does this pattern match
    # there, and could you ask the planner for structured output instead (Chapter 3.1)?
    day_pattern = re.compile(r'Day (\d+)[:\s]+(.*?)(?=Day \d+|$)', re.DOTALL)
    days = day_pattern.findall(plan_text_without_final_answer)
    if not days:
        event = Event()
        event.add('summary', "Travel Itinerary")
        event.add('description', plan_text_without_final_answer)
        event.add('dtstart', start_date.date())
        event.add('dtend', start_date.date())
        event.add("dtstamp", datetime.now())
        cal.add_component(event)
    else:
        for day_num, day_content in days:
            day_num = int(day_num)
            current_date = start_date + timedelta(days=day_num - 1)
            event = Event()
            event.add('summary', f"Day {day_num} Itinerary")
            event.add('description', day_content.strip())
            event.add('dtstart', current_date.date())
            event.add('dtend', current_date.date())
            event.add("dtstamp", datetime.now())
            cal.add_component(event)
    return cal.to_ical()

ics_bytes = generate_ics_content(planner_outputs["verified"])
with open("manali_itinerary.ics", "wb") as ics_file:
    ics_file.write(ics_bytes)

event_count = ics_bytes.decode("utf-8").count("BEGIN:VEVENT")
print(f"Saved manali_itinerary.ics with {event_count} events for a {number_of_days}-day plan.")
print("Download it from the file panel and import it into your calendar.")

## 9. GenAI, AI agents and agentic AI

You have now built all three. The session document compares them like this.

| Feature | GenAI (creative engine) | AI agents (task executors) | Agentic AI (true collaborators) |
|---|---|---|---|
| Core ability | Generates content (text, image, code) | Understands goals, uses tools, takes actions | Plans, adapts, collaborates, learns from experience |
| Trigger | Needs a prompt every time | Executes when given a clear task | Self-manages broad, vague or long-term goals |
| Memory | None (one-shot outputs) | Limited (may hold short-term context) | Long-term, evolving memory (semantic, vector databases) |
| Example task | Write a poem | Book a flight and add it to the calendar | Build, test and deploy a full recipe website |
| Human role | You must instruct every step | Guiding: set a task, the agent executes | Partnering: co-create, iterate and refine together |
| Analogy | A talented writer waiting for prompts | A personal assistant who follows orders | A project manager and co-worker who can think and adapt |
| In this notebook | Chapter 1; 8.1 | The concierge (4.5), FlyBot 2.0 (4.7), the travel agent (8.3) | The four-role planner (8.4), in its simplest form |

Two things to remember when you use these words. First, the same Gemini model sits inside all three columns; the difference is the tools, loop and memory built around it. Second, "learns from experience" and "long-term memory" happen only if someone builds them, as you did with the `lessons` list in 3.4. The model itself does not change between runs.

## 10. Conclusions

**The journey from LLMs to AI agents is a move from reactive to proactive.** An LLM answers prompts and cannot act. Give it tools and a loop and it becomes a goal-driven system that plans, decides and carries out multi-step actions, as the concierge did on Tuesday.

**Agents evolved in stages, each overcoming the last one's limit.** Rule-based agents were rigid. Reactive agents coped with change but could not plan. Deliberative agents planned but were slow and bound to their map. Learning agents adapted with experience. Domain-specific agents were expert but fenced in. You rebuilt each idea with an LLM: understanding, memory, planning, learning from feedback.

**Modern AI agents fuse all these capabilities.** Perception, reasoning, planning and learning combine with language understanding, memory and tool use, so one system can carry a complex task from start to finish.

**Tools and memory are what set modern agents apart.** Tools let an agent touch the real world: calendars, flights, hotels, travel guides. Memory, the conversation you send back and the lessons you store, lets it build on what came before.

**Agents suit complex, multi-step, interconnected problems, and workflows suit the rest.** When you can write down the steps in advance, a workflow is more accurate, testable and cheaper. When you cannot, an agent earns its cost.

And one fact no agent or framework changes: **nothing in an agent checks whether its answer is true.** The model only requests actions; your code performs them; and the only facts you can trust are those you can trace to a tool result. Agents remove plumbing, not decisions. You still choose the tools, write their descriptions, set the step limits, decide which actions need a human to confirm, and decide how to check the output.

## 11. Homework

Each item has a `# TODO: HWn` comment at the exact line it concerns. They describe the problem and ask a question; working out the answer is the exercise.

| # | Level | Task | Where |
|---|---|---|---|
| HW1 | Easy | "Actually, make it 8 pm": write the missing tool | 4.5, goal B |
| HW2 | Medium | Make `add_calendar_event` reject bad dates in a way the model can learn from | 4.1 |
| HW3 | Medium | Stop a wrong tool name or argument from crashing the agent | 4.5, `run_agent` |
| HW4 | Easy | Give the planner the preferences | 8.4, orchestrator |
| HW5 | Medium | Let the QA role see the research | 8.4, `qa_checker` |
| HW6 | Hard | Turn QA into an evaluator-optimizer loop | 8.4, orchestrator |
| HW7 | Easy | Fix the calendar export, ideally with structured output | 8.6 |
| HW8 | Medium | Give the concierge its coffee pages back | below |
| HW9 | Medium | Parallelization from a photo | below |
| HW10 | Easy | Passive or proactive? | below |

**HW8 [Medium]: the concierge forgot its coffee.** The agent in Chapter 4 can manage Prasad's calendar but can no longer answer "is the pink café quiet enough for a first date?", which the Session 5 concierge could. Write a `search_coffee_pages(question: str) -> str` tool that queries your Session 5 Pinecone index, and add it to `calendar_tools`. Read the embedding dimension from the index instead of typing it. What should the docstring say so the model uses this tool for coffee questions and not for dates?

**HW9 [Medium]: materials from a photo.** Gemini can read images. Photograph any small object at home (a bottle cap, a hinge, a phone case) and redo 6.2 with the image instead of a text description, adding Elastomer and Wood from the session document's criteria. Does the recommendation change? How many calls does it cost?

**HW10 [Easy]: passive or proactive?** Rewrite the proactive system prompt in Chapter 2 so the assistant raises exactly one concern and asks exactly one question. Then give it a trip to Goa in July. What does it notice?

**Suggested order.** HW10 first (prompting only). Then HW1, HW2 and HW3 together, since all three are about the agent loop and what tools send back. Then the travel planner: HW4, HW7, HW5, and HW6 last, because an evaluator needs a QA step that can see the research. HW8 and HW9 connect this session to earlier ones; do them when the rest works.

## 12. Quick reference

| Idea | One line | Code you wrote |
|---|---|---|
| Agent loop | Perceive, reason and plan, act, check the result, repeat | Chapter 4 |
| The key question | Who decides the next step? In a workflow your code; in an agent the model | Chapters 5 to 8 |
| Structured output | Get a dictionary back instead of free text | `llm.with_structured_output(schema)` |
| Memory | Whatever you send back to the model | a list of messages; the `lessons` list |
| Planning | Ask for steps first, act later, replan on surprises | `planner_llm` in 3.3 |
| Tool | A Python function with a clear name, type hints and docstring | `get_today`, `check_calendar`, ... |
| Tool call | The model's request to run a tool; nothing runs until your code runs it | `reply.tool_calls` |
| Tool result | Sent back with the matching id | `ToolMessage(content=..., tool_call_id=...)` |
| Agent, long form | The loop by hand, with a step limit | `run_agent(...)` |
| Agent, short form | The same loop from LangChain | `create_agent(model, tools, system_prompt)` |
| Evaluator-optimizer | Produce, grade, feed back, until accepted | bio writer, 6.1 |
| Parallelization | Independent sub-tasks, then aggregate | `llm.batch(...)`, 6.2 |
| Chaining | One focused step after another | write then revise, 6.3 |
| Routing | Classify, then one specialised path | genre router, 6.4 |
| LLM-as-judge | Use a model to check another model's output against a source | grounding check, 8.5 |